<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.04 Getting Started with Raspberry Pi

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>

### 1. Introduction

In this hands-on exercise, we will learn how to set up and connect to a Raspberry Pi for remote access. The goal is to prepare your Pi so you can work with it from your laptop without needing a dedicated screen, keyboard, or mouse.

### 2. Setup

We will use a **Raspberry Pi 5 with 8 GB RAM** as the host machine for our Local LLMs.

<center>
<img src="../../.admin/notebooks/picture/raspberrypi.png" 
     alt="Raspberry Pi 5" 
     width="60%" 
     style="filter: grayscale(80%);">
</center>

The Raspberry Pi is equipped with a lightweight headless OS and runs **[Ollama](https://ollama.com/)**, a tool that simplifies deploying and managing LLMs locally. A headless OS runs without a graphical user interface (GUI). It is managed remotely via command line, SSH and APIs. This makes it efficient, lightweight, and ideal for various domains where keyboards and monitors are not needed, such as servers, IoT devices, and embedded systems.

**If you encounter any issues, do not hesitate to ask the teaching staff for assistance.**



#### 2.1 Connect power and laptop
The Raspberry Pi requires a power connection via USB-C. For this, it is recommended to use the official 27 W USB-C power supply.
Connect this to the USB-C port on the Raspberry Pi. The LED should light up once plugged in (first red, then green, and after around ten seconds it starts flickering. Give it another few seconds before it finally remains on as a green light. The flickering indicates read-write activities during the startup). 

We connect the Raspberry Pi via Ethernet to our laptop. For this, you can either use (1) the provided Ethernet cable with (or without) a USB-C adapter or (2) the provided RJ45 to USB-C cable.

#### 2.2 Reachability test: can the laptop communicate with the Pi?

<div class="alert alert-block alert-info">
Quickly run the test below to find out if the physical connection is working!</div>

We now test that the Raspberry Pi is reachable. Below, we define the connection parameters for the Raspberry Pi. These include IP address, username, and password, which will allow us to connect via SSH.

In [ ]:
raspberry_pi_conf = {
    "ip_addr": "172.31.250.1",
    "username": "cas",
    "password": "password"
}

In the next cell, we test if the Raspberry Pi is reachable over the network. This confirms that the physical connection is working correctly.

In [ ]:
# check if the Raspberry Pi is reachable via ping
import subprocess
import platform

param = "-n" if platform.system().lower() == "windows" else "-c"
command = ["ping", param, "1", raspberry_pi_conf["ip_addr"]]

try:
    output = subprocess.check_output(command, stderr=subprocess.STDOUT, universal_newlines=True)
    print(f"✅ {raspberry_pi_conf['ip_addr']} is reachable")
except subprocess.CalledProcessError:
    print(f"❌ {raspberry_pi_conf['ip_addr']} is NOT reachable")


#### 2.3 Known issues (hopefully this section can be ignored)

<div class="alert alert-block alert-success">
Ignore these issues if you can connect to the Pi. Try to skip this part and continue with the next section.
</div>

Known issues are listed below. If you encounter any of these issues, or something unexpected happens, please reach out. We are happy to help.

- **SSH blocked through firewall:** SSH at times can be blocked by our firewall. However, since we initiate this outbound connection there should be no need for any specific settings. Be aware though that firewalls or security software on your laptop can still interfere with SSH traffic, preventing this setup from working reliably.

- **Possible IP address conflicts**: While highly unlikely, there is a small chance of IP address conflicts. Each Raspberry Pi acts as a DHCP server, assigning IP addresses within its own isolated network. This setup allows us to use a fixed IP for SSH access without needing any configuration changes on your laptop. The server operates on the subnet `172.31.250.0/24` (chosen specifically because it is very uncommon), but if your home router happens to use the same subnet, you may experience unexpected network behavior when using the Pi at home. If this happens, connecting your laptop to a mobile hotspot (e.g. from your phone) during the usage of the Raspberry Pi can resolve the conflict.

- **Internet loss after connecting the Pi:** While not expected, there exists an issue where users may lose internet connectivity after plugging in the Pi. This happens because most operating systems give higher priority to wired (Ethernet) connections over Wi-Fi. It is good practice to revert back any changes once we have finished working with the Pi. Taking screenshots of the current setup helps.
  - **macOS**: adjust the *service order* so that Wi-Fi is listed above Ethernet. [See Apple's guide](https://support.apple.com/guide/mac-help/change-order-network-services-mac-mchlp2711/mac)
  - **Windows**: disable the automatic interface metric and assign Wi-Fi a *lower* metric value than Ethernet. [See TenForum's guide](https://www.tenforums.com/tutorials/92180-change-network-adapter-connection-priorities-windows-10-a.html)

- **Missing tools:** In general, make sure you have downloaded the needed tools for this course: If you are working with this notebook, you have already walked through [`00.1_Check_Course_Environment.ipynb`](../00_Intro/00.1_Check_Course_Environment.ipynb), which contains guidelines on what tools we need.



### 3 Login via SSH

We use an SSH connection to communicate with the Pi. Open a terminal on your machine and try to connect to the Raspberry Pi with Secure Shell (SSH). SSH is a network protocol used for remotely accessing other computers over a network. Run the following commands on your terminal:

```sh
  ping 172.31.250.1
  ssh cas@172.31.250.1
```

When the `ssh` command is called for the first time, a security warning might show up as a new device is about to be accessed. Confirm that you want to continue connecting. Then, upon request, enter "password" and you should be logged in. You will see a Linux Debian shell. Everything you do from now on happens on the Pi. For example, call your first program on the Pi:

```sh
  ollama --version
```
You should see something like what is indicated in the image below.

<center>
<img src="../../.admin/notebooks/picture/raspberry_login.png" 
     alt="Raspberry Pi 5 login" 
     width="60%" 
     style="filter: grayscale(10%);">
</center>


### 4 Ollama
Ollama ([https://ollama.com](https://ollama.com/)) is an open-source runtime for open LLMs. It simplifies model execution by hiding complex setup steps and provides a clean interface to configure, run, and interact with models. To customize this behavior, Ollama uses **Modelfiles**, which define a base model along with persistent parameters. An example of such a file is `custom_model` in the base directory.

#### 4.1 Command overview
Here are the most important commands you will need:

- **Model Management**
  - `ollama list` → list all available models  
  - `ollama show <MODEL>` → display model details (parameters, size, etc.)
  - `ollama create <MODEL> -f <MODELFILE>` → creates a model based on the given modelfile
  - `ollama rm <MODEL>` → deletes the given model

- **Runtime Control**
  - `ollama serve` → manually start the Ollama server *(This service should start automatically on boot. This command blocks the terminal if it is not run in the background.)*
  - `ollama run <MODEL>` → start a session with the specified model  
  - `ollama stop <MODEL>` → stop the running model  
  - `ollama ps` → list running models (you should see at most one active model)  

- **Helpful Flags**
  - `--verbose` → show detailed logs about model loading and requests  
  - `--keepalive <DURATION>` → keep a model in memory after use (default: 5m).  
    Example: `--keepalive 0` unloads immediately after stopping.  
  - `--help` → provide detailed usage info for any command  

- **API**
  Ollama also exposes a **local REST API** (default: `http://localhost:11434`) so you can send requests programmatically. This makes it easier to quickly change parameters for single prompts on the fly. For further information, check out the [official documentation](https://docs.ollama.com/api).<br>
    Example (using `curl` with `jq` for a more readable output):  <br>

    ```sh
    curl http://localhost:11434/api/generate -d '{
        "model": "gemma3:270m",
        "stream": false,
        "prompt": "Hello, world!"
    }' | jq
    ``` 

- **Chat CLI**
    After starting a model and entering a chat, we have a few useful commands available:
    - `/?` → show an overview of available commands
    - `/bye` → exit the chat (alternative to CTRL+D)
    - `/show <OPTION>` → display details for the chosen option


When running models, Ollama first loads them into memory before making them available for prompting. Since these models are computationally expensive and the Raspberry Pi has limited memory, we have to make sure that no two large models are loaded at the same time. To manage this, you can either stop a specific model manually or use the provided custom command `ollama-stop`, which stops all running models.

#### 4.2 First steps
<div class="alert alert-block alert-info">

1. Connect to the Pi via SSH.<br>
2. Explore the newly-learned commands, use the `--help` flag to get more insight into the `run` and `stop` commands.<br>
3. Choose one available model and display some of its details.<br>
4. Experiment with the model. After you are done, exit the chat and confirm that the model has stopped running.
</div>

Try, for example:
```sh
  ollama run deepseek-r1:1.5b --verbose
```

#### 4.3 Monitor the system resources
As demonstrated in the live demo, we can also monitor the Raspberry Pi's resource usage while running LLMs. There are many ways to do so, but using `htop` is one of the quickest and simplest ways.

<center>
<img src="../../.admin/notebooks/picture/htop.png" 
     alt="Raspberry Pi 5: htop" 
     width="60%">

Visit [htop Explained Visually](https://codeahoy.com/2017/01/20/hhtop-explained-visually/) to learn more.
</center>

<div class="alert alert-block alert-info">

1. Connect to the Pi via SSH.<br>
2. Start monitoring resource usage by running `htop`<br>
3. Open a second terminal window (for example in split screen) and start another SSH session to the Pi<br>
4. With both terminals in view, experiment with a model while monitoring the system resources in real time.
</div>

---
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Did you encounter any difficulties during the setup?** Answer:

(add your answer here)

**Get comfortable with the Raspberry Pi's connectors. Which are present? How many? What purpose does each fulfill?** Answer:

(add your answer here)

**Does the Raspberry Pi get hot? What are the two components that help the device stay cool? How do they work? What would happen without them?** Answer:

*Hint: active vs passive*

(add your answer here)

**How many models are available locally via Ollama?** Answer:

(add your answer here)


**Which model did you run? Look up the model info: what does it say?** Answer:

(add your answer here)

**Run a model with the `--verbose` flag. Try to understand the output. What do you see?** Answer:

(add your answer here)

**Try out the API to prompt a model. How does this work? Any drawbacks? Advantages?** Answer:

(add your answer here)

**Take a closer look at the API output. Calculate the tokens per second for the prompt evaluation and for the response generation.** Answer:

*Hint: visit the API documentation to find out how to calculate these metrics* 

(add your answer here)

**Take a closer look at the modelfile, without changing it. Which model acts as the base? How does it get modified?** Answer:

*Hint: `cat <FILENAME>`*

(add your answer here)

**Run a model and monitor the resource usage: how high is the utilization across all cores during text generation? How high is it once the inference has completed?** Answer:

(add your answer here)

**How much memory does your chosen model occupy once it has been loaded?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)